In [1]:
import pandas as pd
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load datasets
movies = pd.read_csv('../data/movies.csv')
ratings = pd.read_csv('../data/ratings.csv')

# Clean genres
movies['genres'] = movies['genres'].str.replace('|', ' ', regex=False)

# Create rating statistics
rating_stats = ratings.groupby('movieId').agg(
    avg_rating=('rating', 'mean'),
    rating_count=('rating', 'count')
).reset_index()

movies = movies.merge(rating_stats, on='movieId', how='left')

# Keep only movies with at least 20 ratings
movies = movies[movies['rating_count'] >= 20]
movies = movies.reset_index(drop=True)

# TF-IDF + Cosine Similarity
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres'])

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

# Normalize rating + popularity
movies['rating_norm'] = (
    movies['avg_rating'] - movies['avg_rating'].min()
) / (
    movies['avg_rating'].max() - movies['avg_rating'].min()
)

movies['count_norm'] = (
    movies['rating_count'] - movies['rating_count'].min()
) / (
    movies['rating_count'].max() - movies['rating_count'].min()
)

# Save Models
pickle.dump(cosine_sim, open('../models/cosine_sim.pkl', 'wb'))
pickle.dump(movies, open('../models/movies.pkl', 'wb'))
pickle.dump(indices, open('../models/indices.pkl', 'wb'))

print("Models saved successfully!")


Models saved successfully!


In [2]:
import pickle

# Save cosine similarity matrix
with open('../models/cosine_sim.pkl', 'wb') as f:
    pickle.dump(cosine_sim, f)

# Save processed movies dataframe
with open('../models/movies.pkl', 'wb') as f:
    pickle.dump(movies, f)

# Save title index mapping
with open('../models/indices.pkl', 'wb') as f:
    pickle.dump(indices, f)

print("Models saved successfully!")


Models saved successfully!
